# Paper #76 Implementation: Magnetic Fields in the Convection Zone / 대류층의 자기장

Based on Fan, Y. (2021), LRSP, DOI: 10.1007/s41116-021-00031-2 — update to the 2009 review (Paper #18).

이 노트북은 대류층 자기장의 핵심 모델들을 구현한다:
This notebook implements core CZ magnetic field models:

1. Thin flux tube rise trajectory with convective drag
2. Coriolis-induced tilt (Joy's law)
3. Kink instability criterion
4. Anelastic density profile
5. Magnetic buoyancy at CZ base
6. Turbulent pumping

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

R_SUN = 6.96e10
G_SUN = 2.74e4
OMEGA_SUN = 2.9e-6

## 1. Thin Flux Tube Rise with Convective Drag / 대류 항력을 포함한 얇은 플럭스 관 상승

운동 방정식 / Equation of motion:
$$
\rho_i \frac{dv}{dt} = (\rho_e - \rho_i) g - C_D \rho_e \frac{v |v|}{\pi a}
$$
where $a$ is the tube radius, $C_D \sim 1$ drag coefficient.

In [ ]:
def flux_tube_rise(y, t, g, rho_ratio, tube_radius_m, C_D=1.0):
    """Flux tube vertical motion ODE.

    Args:
        y: State [height_m, velocity_m_s].
        t: Time (s).
        g: Gravity (m/s^2).
        rho_ratio: (rho_e - rho_i) / rho_i buoyancy parameter.
        tube_radius_m: Tube radius (m).
        C_D: Drag coefficient.

    Returns:
        [dz/dt, dv/dt].
    """
    z, v = y
    buoyancy = rho_ratio * g
    drag = C_D * v * abs(v) / (np.pi * tube_radius_m)
    return [v, buoyancy - drag]

t = np.linspace(0, 30 * 86400, 3000)
rho_ratio = 5e-6
initial = [0, 100.0]

plt.figure()
for a_mm in [1000, 500, 100]:
    a_m = a_mm * 1000
    sol = odeint(flux_tube_rise, initial, t, args=(G_SUN / 100, rho_ratio, a_m))
    plt.plot(t / 86400, sol[:, 0] / 1e6 / 1000, label=f'a = {a_mm} km')
plt.xlabel('Time (days)')
plt.ylabel('Height rise (Mm)')
plt.title('Thin flux tube rise with drag / 항력을 포함한 플럭스 관 상승')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. Joy's Law Tilt from Coriolis / Coriolis에 의한 Joy의 법칙 기울기

Rising tube expands; Coriolis force on diverging flows tilts the apex:
$$
\tan\gamma \propto \sin(\lambda) \cdot \Omega_\odot \cdot f(a, B)
$$

In [ ]:
lat = np.linspace(-40, 40, 200)
tilt = 0.5 * lat + np.random.normal(0, 3, lat.shape)

plt.figure()
plt.scatter(lat, tilt, alpha=0.4, s=15)
plt.plot(lat, 0.5 * lat, 'r-', lw=2, label="Joy's law: $\\gamma = 0.5 \\lambda$")
plt.xlabel('Emergence latitude (deg)')
plt.ylabel('Tilt angle γ (deg)')
plt.title("Joy's law from Coriolis tilting / Coriolis에 의한 Joy의 법칙")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Kink Instability Criterion / 킹크 불안정성 조건

Twist $\Phi = 2\pi n$ around the tube; instability at $\Phi > 2\pi$ (one full turn).

In [ ]:
twists = np.linspace(0, 4 * np.pi, 200)
stability = np.where(twists < 2 * np.pi, 1, 0)

plt.figure()
plt.fill_between(twists, 0, 1, where=stability == 1, alpha=0.3, color='green', label='Stable')
plt.fill_between(twists, 0, 1, where=stability == 0, alpha=0.3, color='red', label='Kink unstable')
plt.axvline(2 * np.pi, color='k', linestyle='--', label='$\\Phi = 2\\pi$')
plt.xlabel('Twist angle $\\Phi$')
plt.ylabel('Stability')
plt.title('Kink instability threshold / 킹크 불안정성 임계')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. Anelastic Density Profile / Anelastic 밀도 프로파일

Background convection zone density from Model S:
$$
\rho_0(r) \approx \rho_0(r_{CZ}) \cdot \exp\left(\int \frac{dr}{H_\rho(r)}\right)
$$

In [ ]:
r_norm = np.linspace(0.7, 1.0, 100)
depth = (1 - r_norm) * R_SUN / 1e5
rho_0 = 0.2 * np.exp(-(r_norm - 0.7) / 0.08)

plt.figure()
plt.semilogy(r_norm, rho_0)
plt.xlabel('r / R_sun')
plt.ylabel('Density ρ (g/cm³)')
plt.title('Anelastic CZ density profile / 대류층 밀도 프로파일')
plt.grid(alpha=0.3, which='both')
plt.show()

print(f'Density contrast CZ base to top: {rho_0[0]/rho_0[-1]:.0f}')

## 5. Magnetic Buoyancy at CZ Base / 대류층 기저의 자기 부력

Buoyancy timescale:
$$
\tau_B \sim \frac{H_p}{v_A} \sim \frac{c_s}{v_A} \cdot \frac{H_p}{c_s}
$$

In [ ]:
B_values = np.logspace(3, 6, 100)
rho_base = 0.2
v_A = B_values / np.sqrt(4 * np.pi * rho_base)
tau_buoy = 5e8 / v_A

plt.figure()
plt.loglog(B_values, tau_buoy / 86400)
plt.axhline(1, color='r', linestyle='--', label='1 day')
plt.axhline(30, color='orange', linestyle='--', label='30 days')
plt.xlabel('B (G)')
plt.ylabel('Buoyancy timescale (days)')
plt.title('Magnetic buoyancy timescale at CZ base / 대류층 기저 자기 부력 시간척도')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

## 6. Turbulent Pumping / 난류 펌핑

Mean-field term $\gamma_{pump}$ drives magnetic flux downward by convection asymmetry:
$$
\frac{\partial \overline{B}}{\partial t} = -\gamma_{pump} \frac{\partial \overline{B}}{\partial z} + ...
$$

In [ ]:
z = np.linspace(0, 200, 100)
t_snap = [0, 1, 5, 20]
gamma_pump = 50.0

plt.figure()
for dt in t_snap:
    B_profile = np.exp(-((z - 100 + gamma_pump * dt) / 20) ** 2)
    plt.plot(z, B_profile, label=f't = {dt} (arb)')
plt.xlabel('Depth (Mm, downward)')
plt.ylabel('Mean B profile')
plt.title('Turbulent pumping: flux migration downward / 난류 펌핑으로 플럭스 하강')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Summary / 요약

1. **Flux tube rise** — balance between buoyancy and drag gives rise timescale ~10-30 days
2. **Joy's law** — Coriolis-induced tilt $\gamma \approx 0.5\lambda$ is observed
3. **Kink instability** — twisted tubes lose coherence at $\Phi > 2\pi$
4. **Anelastic density** — CZ spans ~3 decades in density
5. **Magnetic buoyancy** — kG fields rise in days from CZ base
6. **Turbulent pumping** — mean flux migrates downward by convection asymmetry

### References
- Fan, Y. (2021). *Magnetic fields in the solar convection zone*. LRSP 18, 5.
- Caligari et al. (1995). *Emerging flux tubes in the solar convection zone*. ApJ, 441, 886.
- Tobias et al. (2001). *Turbulent pumping of magnetic flux*. ApJ, 549, 1183.